In [ ]:
!pip install requests pandas numpy openpyxl

In [6]:
# ============================================================
# 台股自動選股系統 V1 + V2 + V3 + V4 + V5 完整版
#
# V1 技術面        100分
# V2 法人籌碼面     65分 → V5標準化100分
# V3 基本面        100分
# V4 風險動能      100分
# V5 最終決策      100分
#
# V5權重：
# V1 技術       30%
# V2 籌碼       20%
# V3 基本面     30%
# V4 風險動能   20%
#
# 資料來源：
# V1/V4：FinMind 歷史價格（只抓一次）
# V2：TWSE + TPEx 官方
# V3：MOPS + TWSE + TPEx 官方
# ============================================================


# ============================================================
# 0. 安裝
# ============================================================

!pip -q install FinMind openpyxl


# ============================================================
# 1. 載入套件
# ============================================================

import io
import time
import warnings
import requests
import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from getpass import getpass

from FinMind.data import DataLoader

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter


warnings.filterwarnings("ignore")


# ============================================================
# 2. 基本設定
# ============================================================

FINMIND_TOKEN = getpass(
    "請輸入 FinMind Token："
).strip()


最低成交股數 = 1_000_000      # 1000張
歷史天數 = 120
法人天數 = 5

批次大小 = 30
補抓等待 = 0.35

最低產業樣本數 = 10
前幾名 = 20


HEADERS = {
    "User-Agent":
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "Chrome/153 Safari/537.36",

    "Accept":
        "application/json,text/plain,*/*"
}


今天 = datetime.now(
    ZoneInfo("Asia/Taipei")
).date()


# ============================================================
# 3. 產業分類
# ============================================================

產業對照 = {

    "01": "水泥工業",
    "02": "食品工業",
    "03": "塑膠工業",
    "04": "紡織纖維",
    "05": "電機機械",
    "06": "電器電纜",

    "08": "玻璃陶瓷",
    "09": "造紙工業",
    "10": "鋼鐵工業",
    "11": "橡膠工業",
    "12": "汽車工業",

    "14": "建材營造",
    "15": "航運業",
    "16": "觀光餐旅",
    "17": "金融保險",
    "18": "貿易百貨",
    "19": "綜合",
    "20": "其他",

    "21": "化學工業",
    "22": "生技醫療業",
    "23": "油電燃氣業",

    "24": "半導體業",
    "25": "電腦及週邊設備業",
    "26": "光電業",
    "27": "通信網路業",
    "28": "電子零組件業",
    "29": "電子通路業",
    "30": "資訊服務業",
    "31": "其他電子業",

    "35": "綠能環保",
    "36": "數位雲端",
    "37": "運動休閒",
    "38": "居家生活"
}


# ============================================================
# 4. 共用工具
# ============================================================

def 數字(x):

    if x is None:
        return np.nan

    text = str(x).strip()

    if text in [
        "",
        "-",
        "--",
        "---",
        "—",
        "－",
        "N/A",
        "nan",
        "None",
        "<NA>"
    ]:
        return np.nan

    text = (
        text
        .replace(",", "")
        .replace("%", "")
        .replace("+", "")
        .replace(" ", "")
        .replace("\u3000", "")
    )

    try:
        return float(text)

    except:
        return np.nan


def 安全除法(a, b):

    a = pd.to_numeric(
        a,
        errors="coerce"
    )

    b = pd.to_numeric(
        b,
        errors="coerce"
    )

    result = a / b

    return result.replace(
        [np.inf, -np.inf],
        np.nan
    )


def 民國日期(date):

    return (
        f"{date.year - 1911}/"
        f"{date.month:02d}/"
        f"{date.day:02d}"
    )


def 取得JSON(
    url,
    params=None
):

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=60
    )

    response.raise_for_status()

    return response.json()


def 取得CSV(url):

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=60
        )

        response.raise_for_status()

        content = response.content


        for encoding in [
            "utf-8-sig",
            "utf-8",
            "big5"
        ]:

            try:

                text = content.decode(
                    encoding
                )

                df = pd.read_csv(
                    io.StringIO(text),
                    dtype=str
                )

                df.columns = [
                    str(c)
                    .replace("\ufeff", "")
                    .replace("\u3000", "")
                    .strip()
                    for c in df.columns
                ]

                return df

            except:
                pass


        return pd.DataFrame()


    except Exception as e:

        print(
            "CSV下載失敗：",
            url,
            str(e)[:120]
        )

        return pd.DataFrame()


def 找欄位(
    df,
    候選名稱
):

    if df.empty:
        return None


    for name in 候選名稱:

        if name in df.columns:
            return name


    for name in 候選名稱:

        for col in df.columns:

            if name in str(col):
                return col


    return None


# ============================================================
# 5. TWSE 每日行情
# ============================================================

def 取得上市行情(date):

    url = (
        "https://www.twse.com.tw/"
        "rwd/zh/afterTrading/MI_INDEX"
    )


    params = {

        "date":
            date.strftime("%Y%m%d"),

        "type":
            "ALLBUT0999",

        "response":
            "json"
    }


    data = 取得JSON(
        url,
        params
    )


    target = None


    for table in data.get(
        "tables",
        []
    ):

        fields = table.get(
            "fields",
            []
        )

        if (
            "證券代號" in fields
            and
            "證券名稱" in fields
            and
            "成交股數" in fields
        ):

            target = table
            break


    if target is None:
        return pd.DataFrame()


    df = pd.DataFrame(
        target["data"],
        columns=target["fields"]
    )


    df = df.rename(
        columns={

            "證券代號":
                "股票代號",

            "證券名稱":
                "股票名稱",

            "成交股數":
                "成交股數"
        }
    )


    df = df[
        [
            "股票代號",
            "股票名稱",
            "成交股數"
        ]
    ]


    df["股票代號"] = (
        df["股票代號"]
        .astype(str)
        .str.strip()
    )


    df["成交股數"] = (
        df["成交股數"]
        .apply(數字)
    )


    df["市場"] = "上市"


    return df


# ============================================================
# 6. TPEx 每日行情
# ============================================================

def 取得上櫃行情(date):

    url = (
        "https://www.tpex.org.tw/"
        "web/stock/aftertrading/"
        "daily_close_quotes/"
        "stk_quote_result.php"
    )


    params = {

        "l":
            "zh-tw",

        "o":
            "json",

        "d":
            民國日期(date),

        "s":
            "0,asc,0"
    }


    data = 取得JSON(
        url,
        params
    )


    result = []


    for row in data.get(
        "aaData",
        []
    ):

        if len(row) < 9:
            continue


        result.append({

            "股票代號":
                str(row[0]).strip(),

            "股票名稱":
                str(row[1]).strip(),

            "成交股數":
                數字(row[8]),

            "市場":
                "上櫃"
        })


    return pd.DataFrame(
        result
    )


# ============================================================
# 7. 找最近交易日
# ============================================================

def 找最近交易日():

    for i in range(12):

        date = (
            今天
            -
            timedelta(days=i)
        )


        if date.weekday() >= 5:
            continue


        try:

            tw = 取得上市行情(
                date
            )

        except:

            tw = pd.DataFrame()


        try:

            tp = 取得上櫃行情(
                date
            )

        except:

            tp = pd.DataFrame()


        frames = []


        if not tw.empty:
            frames.append(tw)


        if not tp.empty:
            frames.append(tp)


        if frames:

            return (
                date,
                pd.concat(
                    frames,
                    ignore_index=True
                )
            )


    raise RuntimeError(
        "找不到最近交易日"
    )


# ============================================================
# 8. 股票第一層篩選
# ============================================================

最新交易日, 市場資料 = (
    找最近交易日()
)


print(
    "\n======================================"
)

print(
    "最新交易日：",
    最新交易日
)

print(
    "======================================"
)


市場資料["股票代號"] = (
    市場資料[
        "股票代號"
    ]
    .astype(str)
    .str.strip()
)


# 只留4碼普通股票
市場資料 = 市場資料[
    市場資料[
        "股票代號"
    ].str.match(
        r"^\d{4}$",
        na=False
    )
].copy()


# 排除00開頭ETF
市場資料 = 市場資料[
    ~市場資料[
        "股票代號"
    ].str.startswith(
        "00"
    )
].copy()


# 成交量 >=1000張
活躍股票 = 市場資料[
    市場資料[
        "成交股數"
    ]
    >=
    最低成交股數
].copy()


活躍股票 = (
    活躍股票
    .drop_duplicates(
        "股票代號"
    )
    .sort_values(
        "成交股數",
        ascending=False
    )
)


股票清單 = (
    活躍股票[
        "股票代號"
    ]
    .tolist()
)


print(
    "成交量 >= 1000張：",
    len(股票清單),
    "檔"
)


# ============================================================
# 9. FinMind 歷史資料
#
# V1 + V4 共用
# ============================================================

print(
    "\n======================================"
)

print(
    "V1 + V4 共用 FinMind 歷史資料"
)

print(
    "======================================"
)


api = DataLoader()


api.login_by_token(
    api_token=
        FINMIND_TOKEN
)


開始日期 = (
    最新交易日
    -
    timedelta(
        days=歷史天數
    )
).strftime(
    "%Y-%m-%d"
)


結束日期 = (
    最新交易日
    +
    timedelta(days=1)
).strftime(
    "%Y-%m-%d"
)


歷史列表 = []

FinMind批次呼叫次數 = 0
FinMind補抓次數 = 0
FinMind額度中止 = False


for i in range(
    0,
    len(股票清單),
    批次大小
):

    batch = 股票清單[
        i:i + 批次大小
    ]


    FinMind批次呼叫次數 += 1


    print(
        f"股價批次 "
        f"{i + 1} ~ "
        f"{min(i + 批次大小, len(股票清單))}"
        f" / {len(股票清單)}"
    )


    try:

        df = api.taiwan_stock_daily(

            stock_id_list=batch,

            start_date=開始日期,

            end_date=結束日期,

            use_async=True
        )


        if (
            df is not None
            and
            not df.empty
        ):

            df["stock_id"] = (
                df["stock_id"]
                .astype(str)
            )


            歷史列表.append(
                df
            )


    except Exception as e:

        error_text = str(e)


        print(
            "批次失敗：",
            error_text[:120]
        )


        if (
            "402" in error_text
            or
            "upper limit"
            in error_text.lower()
            or
            "request limit"
            in error_text.lower()
        ):

            FinMind額度中止 = True

            print(
                "⚠ FinMind額度到達上限"
            )

            break


if not 歷史列表:

    raise RuntimeError(
        "FinMind 沒有取得任何歷史股價"
    )


歷史資料 = pd.concat(
    歷史列表,
    ignore_index=True
)


歷史資料 = (
    歷史資料
    .drop_duplicates(
        [
            "stock_id",
            "date"
        ]
    )
)


# ============================================================
# 10. FinMind漏抓自動補抓
# ============================================================

已抓 = set(
    歷史資料[
        "stock_id"
    ]
    .astype(str)
    .unique()
)


漏抓 = sorted(
    set(股票清單)
    -
    已抓
)


print(
    "\n批次已抓股票：",
    len(已抓)
)


print(
    "需要補抓股票：",
    len(漏抓)
)


補抓列表 = []


if not FinMind額度中止:

    for i, stock_id in enumerate(
        漏抓,
        1
    ):

        FinMind補抓次數 += 1


        print(
            f"[補抓 {i}/{len(漏抓)}] "
            f"{stock_id}",
            end=" "
        )


        try:

            temp = api.taiwan_stock_daily(

                stock_id=stock_id,

                start_date=開始日期,

                end_date=結束日期
            )


            if (
                temp is not None
                and
                not temp.empty
            ):

                temp["stock_id"] = (
                    temp["stock_id"]
                    .astype(str)
                )


                補抓列表.append(
                    temp
                )


                print("成功")


            else:

                print("無資料")


        except Exception as e:

            error_text = str(e)


            print(
                "失敗：",
                error_text[:100]
            )


            if (
                "402" in error_text
                or
                "upper limit"
                in error_text.lower()
                or
                "request limit"
                in error_text.lower()
            ):

                FinMind額度中止 = True


                print(
                    "\n⚠ FinMind額度達上限，停止補抓"
                )

                break


        time.sleep(
            補抓等待
        )


if 補抓列表:

    歷史資料 = pd.concat(
        [
            歷史資料,

            pd.concat(
                補抓列表,
                ignore_index=True
            )
        ],
        ignore_index=True
    )


歷史資料 = (
    歷史資料
    .drop_duplicates(
        [
            "stock_id",
            "date"
        ]
    )
    .sort_values(
        [
            "stock_id",
            "date"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n歷史資料成功股票：",
    歷史資料[
        "stock_id"
    ].nunique(),
    "/",
    len(股票清單)
)


# ============================================================
# 11. 整理價格欄位
# ============================================================

for col in [
    "open",
    "max",
    "min",
    "close",
    "Trading_Volume"
]:

    歷史資料[col] = pd.to_numeric(
        歷史資料[col],
        errors="coerce"
    )


歷史資料["date"] = pd.to_datetime(
    歷史資料["date"]
)


# ============================================================
# 12. V1 + V4 指標一次計算
# ============================================================

def 計算技術與風險指標(g):

    g = (
        g.sort_values(
            "date"
        )
        .copy()
    )


    # ========================================================
    # V1
    # ========================================================

    g["5日均線"] = (
        g["close"]
        .rolling(5)
        .mean()
    )


    g["20日均線"] = (
        g["close"]
        .rolling(20)
        .mean()
    )


    g["60日均線"] = (
        g["close"]
        .rolling(60)
        .mean()
    )


    EMA12 = (
        g["close"]
        .ewm(
            span=12,
            adjust=False
        )
        .mean()
    )


    EMA26 = (
        g["close"]
        .ewm(
            span=26,
            adjust=False
        )
        .mean()
    )


    g["MACD"] = (
        EMA12
        -
        EMA26
    )


    g["MACD訊號線"] = (
        g["MACD"]
        .ewm(
            span=9,
            adjust=False
        )
        .mean()
    )


    g["MACD柱狀體"] = (
        g["MACD"]
        -
        g["MACD訊號線"]
    )


    delta = (
        g["close"]
        .diff()
    )


    gain = (
        delta.clip(
            lower=0
        )
    )


    loss = (
        -delta.clip(
            upper=0
        )
    )


    avg_gain = (
        gain
        .ewm(
            alpha=1 / 14,
            adjust=False
        )
        .mean()
    )


    avg_loss = (
        loss
        .ewm(
            alpha=1 / 14,
            adjust=False
        )
        .mean()
    )


    rs = (
        avg_gain
        /
        avg_loss
    )


    g["14日RSI"] = (
        100
        -
        (
            100
            /
            (
                1 + rs
            )
        )
    )


    g["20日平均成交量"] = (
        g["Trading_Volume"]
        .rolling(20)
        .mean()
    )


    g["量比"] = (
        g["Trading_Volume"]
        /
        g["20日平均成交量"]
    )


    g["前20日最高價"] = (
        g["max"]
        .rolling(20)
        .max()
        .shift(1)
    )


    g["前一日20日均線"] = (
        g["20日均線"]
        .shift(1)
    )


    # ========================================================
    # V4
    # ========================================================

    g["日報酬率"] = (
        g["close"]
        .pct_change()
    )


    g["5日報酬率"] = (
        (
            g["close"]
            /
            g["close"].shift(5)
            -
            1
        )
        *
        100
    )


    g["20日報酬率"] = (
        (
            g["close"]
            /
            g["close"].shift(20)
            -
            1
        )
        *
        100
    )


    g["60日報酬率"] = (
        (
            g["close"]
            /
            g["close"].shift(60)
            -
            1
        )
        *
        100
    )


    g["20日年化波動率"] = (
        g["日報酬率"]
        .rolling(20)
        .std()
        *
        np.sqrt(252)
        *
        100
    )


    # ATR
    前收盤 = (
        g["close"]
        .shift(1)
    )


    TR1 = (
        g["max"]
        -
        g["min"]
    )


    TR2 = (
        g["max"]
        -
        前收盤
    ).abs()


    TR3 = (
        g["min"]
        -
        前收盤
    ).abs()


    TR = pd.concat(
        [
            TR1,
            TR2,
            TR3
        ],
        axis=1
    ).max(
        axis=1
    )


    g["ATR14"] = (
        TR
        .rolling(14)
        .mean()
    )


    g["ATR百分比"] = (
        g["ATR14"]
        /
        g["close"]
        *
        100
    )


    # 均線乖離
    g["MA20乖離率"] = (
        (
            g["close"]
            /
            g["20日均線"]
            -
            1
        )
        *
        100
    )


    g["MA60乖離率"] = (
        (
            g["close"]
            /
            g["60日均線"]
            -
            1
        )
        *
        100
    )


    # 20日最大回撤
    rolling_high = (
        g["close"]
        .rolling(20)
        .max()
    )


    drawdown = (
        g["close"]
        /
        rolling_high
        -
        1
    )


    g["20日最大回撤"] = (
        drawdown
        .rolling(20)
        .min()
        *
        100
    )


    # 成交量穩定程度
    volume_mean = (
        g["Trading_Volume"]
        .rolling(20)
        .mean()
    )


    volume_std = (
        g["Trading_Volume"]
        .rolling(20)
        .std()
    )


    g["20日成交量變異係數"] = (
        volume_std
        /
        volume_mean
    )


    return g


歷史資料 = (
    歷史資料
    .groupby(
        "stock_id",
        group_keys=False
    )
    .apply(
        計算技術與風險指標
    )
)


最新技術資料 = (
    歷史資料
    .sort_values(
        "date"
    )
    .groupby(
        "stock_id"
    )
    .tail(1)
    .copy()
)


# ============================================================
# 13. V1 技術評分
# ============================================================

print(
    "\n======================================"
)

print(
    "V1 技術面"
)

print(
    "======================================"
)


最新技術資料[
    "技術分數"
] = 0


最新技術資料[
    "技術面原因"
] = ""


def 技術加分(
    condition,
    score,
    reason
):

    最新技術資料.loc[
        condition,
        "技術分數"
    ] += score


    最新技術資料.loc[
        condition,
        "技術面原因"
    ] += reason + "、"


技術加分(
    最新技術資料[
        "close"
    ]
    >
    最新技術資料[
        "20日均線"
    ],
    10,
    "股價高於20日均線"
)


技術加分(
    最新技術資料[
        "5日均線"
    ]
    >
    最新技術資料[
        "20日均線"
    ],
    10,
    "5日均線高於20日均線"
)


技術加分(
    最新技術資料[
        "20日均線"
    ]
    >
    最新技術資料[
        "60日均線"
    ],
    15,
    "20日均線高於60日均線"
)


技術加分(
    最新技術資料[
        "20日均線"
    ]
    >
    最新技術資料[
        "前一日20日均線"
    ],
    10,
    "20日均線向上"
)


技術加分(
    最新技術資料[
        "14日RSI"
    ].between(
        50,
        70
    ),
    10,
    "RSI介於50至70"
)


技術加分(
    最新技術資料[
        "MACD"
    ]
    >
    最新技術資料[
        "MACD訊號線"
    ],
    10,
    "MACD多頭"
)


技術加分(
    最新技術資料[
        "MACD柱狀體"
    ]
    >
    0,
    5,
    "MACD柱狀體為正"
)


技術加分(
    最新技術資料[
        "量比"
    ]
    >=
    1.5,
    15,
    "量比達1.5以上"
)


技術加分(
    最新技術資料[
        "close"
    ]
    >
    最新技術資料[
        "前20日最高價"
    ],
    15,
    "突破20日高點"
)


最新技術資料[
    "技術面原因"
] = (
    最新技術資料[
        "技術面原因"
    ]
    .str.rstrip("、")
)


# ============================================================
# 14. V4 風險動能評分
#
# 滿分 = 100
# ============================================================

print(
    "\n======================================"
)

print(
    "V4 風險動能"
)

print(
    "======================================"
)


最新技術資料[
    "風險動能分數"
] = 0


最新技術資料[
    "V4原因"
] = ""


def V4加分(
    condition,
    score,
    reason
):

    最新技術資料.loc[
        condition,
        "風險動能分數"
    ] += score


    最新技術資料.loc[
        condition,
        "V4原因"
    ] += reason + "、"


# 20日報酬 15
V4加分(
    最新技術資料[
        "20日報酬率"
    ].between(
        5,
        20
    ),
    15,
    "20日動能健康"
)


V4加分(
    (
        最新技術資料[
            "20日報酬率"
        ].between(
            0,
            5,
            inclusive="left"
        )
    )
    |
    (
        最新技術資料[
            "20日報酬率"
        ].between(
            20,
            30,
            inclusive="right"
        )
    ),
    8,
    "20日動能尚可"
)


# 60日報酬 15
V4加分(
    最新技術資料[
        "60日報酬率"
    ].between(
        10,
        35
    ),
    15,
    "60日趨勢健康"
)


V4加分(
    (
        最新技術資料[
            "60日報酬率"
        ].between(
            0,
            10,
            inclusive="left"
        )
    )
    |
    (
        最新技術資料[
            "60日報酬率"
        ].between(
            35,
            50,
            inclusive="right"
        )
    ),
    8,
    "60日趨勢尚可"
)


# 波動率 15
V4加分(
    最新技術資料[
        "20日年化波動率"
    ].between(
        15,
        40
    ),
    15,
    "波動率合理"
)


V4加分(
    (
        最新技術資料[
            "20日年化波動率"
        ].between(
            10,
            15,
            inclusive="left"
        )
    )
    |
    (
        最新技術資料[
            "20日年化波動率"
        ].between(
            40,
            55,
            inclusive="right"
        )
    ),
    8,
    "波動率可接受"
)


# ATR 10
V4加分(
    最新技術資料[
        "ATR百分比"
    ].between(
        1,
        4
    ),
    10,
    "ATR風險合理"
)


V4加分(
    最新技術資料[
        "ATR百分比"
    ].between(
        4,
        6,
        inclusive="right"
    ),
    5,
    "ATR偏高但可接受"
)


# 最大回撤 10
V4加分(
    最新技術資料[
        "20日最大回撤"
    ]
    >=
    -8,
    10,
    "20日回撤小"
)


V4加分(
    最新技術資料[
        "20日最大回撤"
    ].between(
        -15,
        -8,
        inclusive="left"
    ),
    5,
    "20日回撤可接受"
)


# MA20乖離 10
V4加分(
    最新技術資料[
        "MA20乖離率"
    ].between(
        2,
        8
    ),
    10,
    "MA20乖離健康"
)


V4加分(
    (
        最新技術資料[
            "MA20乖離率"
        ].between(
            0,
            2,
            inclusive="left"
        )
    )
    |
    (
        最新技術資料[
            "MA20乖離率"
        ].between(
            8,
            15,
            inclusive="right"
        )
    ),
    5,
    "MA20乖離可接受"
)


# MA60乖離 10
V4加分(
    最新技術資料[
        "MA60乖離率"
    ].between(
        3,
        15
    ),
    10,
    "MA60位置健康"
)


V4加分(
    (
        最新技術資料[
            "MA60乖離率"
        ].between(
            0,
            3,
            inclusive="left"
        )
    )
    |
    (
        最新技術資料[
            "MA60乖離率"
        ].between(
            15,
            25,
            inclusive="right"
        )
    ),
    5,
    "MA60乖離可接受"
)


# 成交量穩定度 5
V4加分(
    最新技術資料[
        "20日成交量變異係數"
    ]
    <=
    0.6,
    5,
    "成交量穩定"
)


V4加分(
    最新技術資料[
        "20日成交量變異係數"
    ].between(
        0.6,
        1.0,
        inclusive="right"
    ),
    3,
    "成交量穩定度普通"
)


# 避免短線急漲 10
V4加分(
    最新技術資料[
        "5日報酬率"
    ].between(
        -3,
        10
    ),
    10,
    "短線未過熱"
)


V4加分(
    最新技術資料[
        "5日報酬率"
    ].between(
        10,
        18,
        inclusive="right"
    ),
    5,
    "短線略熱"
)


最新技術資料[
    "V4原因"
] = (
    最新技術資料[
        "V4原因"
    ]
    .str.rstrip("、")
)


# ============================================================
# 15. V2 TWSE法人
# ============================================================

def 取得上市法人(date):

    url = (
        "https://www.twse.com.tw/"
        "rwd/zh/fund/T86"
    )


    params = {

        "response":
            "json",

        "date":
            date.strftime("%Y%m%d"),

        "selectType":
            "ALLBUT0999"
    }


    data = 取得JSON(
        url,
        params
    )


    if data.get(
        "stat"
    ) != "OK":

        return pd.DataFrame()


    df = pd.DataFrame(
        data.get(
            "data",
            []
        ),
        columns=
            data.get(
                "fields",
                []
            )
    )


    if df.empty:
        return df


    code = 找欄位(
        df,
        ["證券代號"]
    )


    foreign = 找欄位(
        df,
        [
            "外陸資買賣超股數",
            "外陸資買賣超"
        ]
    )


    trust = 找欄位(
        df,
        [
            "投信買賣超股數",
            "投信買賣超"
        ]
    )


    dealer = 找欄位(
        df,
        [
            "自營商買賣超股數",
            "自營商買賣超"
        ]
    )


    if any(
        x is None
        for x in [
            code,
            foreign,
            trust,
            dealer
        ]
    ):

        return pd.DataFrame()


    out = pd.DataFrame({

        "股票代號":
            df[code]
            .astype(str)
            .str.strip(),

        "外資淨買超":
            df[foreign]
            .apply(數字),

        "投信淨買超":
            df[trust]
            .apply(數字),

        "自營商淨買超":
            df[dealer]
            .apply(數字)
    })


    out["日期"] = (
        pd.Timestamp(date)
    )


    return out


# ============================================================
# 16. V2 TPEx法人
# ============================================================

def 取得上櫃法人(date):

    url = (
        "https://www.tpex.org.tw/"
        "web/stock/3insti/"
        "daily_trade/"
        "3itrade_hedge_result.php"
    )


    params = {

        "l":
            "zh-tw",

        "o":
            "json",

        "se":
            "EW",

        "t":
            "D",

        "d":
            民國日期(date),

        "s":
            "0,asc"
    }


    data = 取得JSON(
        url,
        params
    )


    result = []


    for row in data.get(
        "aaData",
        []
    ):

        if len(row) < 23:
            continue


        result.append({

            "股票代號":
                str(row[0]).strip(),

            "外資淨買超":
                數字(row[10]),

            "投信淨買超":
                數字(row[13]),

            "自營商淨買超":
                數字(row[22]),

            "日期":
                pd.Timestamp(date)
        })


    return pd.DataFrame(
        result
    )


# ============================================================
# 17. 最近5個法人交易日
# ============================================================

print(
    "\n======================================"
)

print(
    "V2 籌碼面"
)

print(
    "======================================"
)


法人列表 = []
法人日期 = []


d = 最新交易日


while (
    len(法人日期)
    <
    法人天數
):

    if d.weekday() < 5:

        try:

            tw = 取得上市法人(
                d
            )

        except:

            tw = pd.DataFrame()


        time.sleep(1)


        try:

            tp = 取得上櫃法人(
                d
            )

        except:

            tp = pd.DataFrame()


        frames = []


        if not tw.empty:
            frames.append(tw)


        if not tp.empty:
            frames.append(tp)


        if frames:

            法人列表.append(
                pd.concat(
                    frames,
                    ignore_index=True
                )
            )


            法人日期.append(
                d
            )


            print(
                "法人交易日：",
                d
            )


    d -= timedelta(days=1)


if 法人列表:

    法人資料 = pd.concat(
        法人列表,
        ignore_index=True
    )

else:

    法人資料 = pd.DataFrame()


if not 法人資料.empty:

    法人資料 = 法人資料[
        法人資料[
            "股票代號"
        ].isin(
            股票清單
        )
    ].copy()


# ============================================================
# 18. 法人摘要
# ============================================================

def 計算連買(series):

    count = 0


    for value in reversed(
        series
        .fillna(0)
        .tolist()
    ):

        if value > 0:

            count += 1

        else:

            break


    return count


法人摘要列表 = []


if not 法人資料.empty:

    for stock_id, g in (
        法人資料
        .groupby(
            "股票代號"
        )
    ):

        g = (
            g.sort_values(
                "日期"
            )
        )


        foreign = (
            g[
                "外資淨買超"
            ].sum()
        )


        trust = (
            g[
                "投信淨買超"
            ].sum()
        )


        dealer = (
            g[
                "自營商淨買超"
            ].sum()
        )


        法人摘要列表.append({

            "股票代號":
                stock_id,

            "外資近5日買賣超股數":
                foreign,

            "投信近5日買賣超股數":
                trust,

            "自營商近5日買賣超股數":
                dealer,

            "三大法人近5日合計股數":
                foreign
                +
                trust
                +
                dealer,

            "外資連續買超天數":
                計算連買(
                    g[
                        "外資淨買超"
                    ]
                ),

            "投信連續買超天數":
                計算連買(
                    g[
                        "投信淨買超"
                    ]
                ),

            "法人資料天數":
                len(g)
        })


法人摘要 = pd.DataFrame(
    法人摘要列表
)


# ============================================================
# 19. V3 MOPS公司母體
# ============================================================

print(
    "\n======================================"
)

print(
    "V3 MOPS 基本面"
)

print(
    "======================================"
)


上市公司 = 取得CSV(
    "https://mopsfin.twse.com.tw/"
    "opendata/t187ap03_L.csv"
)


上櫃公司 = 取得CSV(
    "https://mopsfin.twse.com.tw/"
    "opendata/t187ap03_O.csv"
)


def 整理公司母體(
    df,
    market
):

    if df.empty:
        return pd.DataFrame()


    code = 找欄位(
        df,
        ["公司代號"]
    )


    name = 找欄位(
        df,
        [
            "公司簡稱",
            "公司名稱"
        ]
    )


    industry = 找欄位(
        df,
        [
            "產業別",
            "產業類別"
        ]
    )


    if code is None:
        return pd.DataFrame()


    out = pd.DataFrame()


    out["股票代號"] = (
        df[code]
        .astype(str)
        .str.strip()
        .str.replace(
            ".0",
            "",
            regex=False
        )
    )


    if name:

        out["公司名稱"] = (
            df[name]
            .astype(str)
            .str.strip()
        )

    else:

        out["公司名稱"] = ""


    if industry:

        original = (
            df[industry]
            .astype(str)
            .str.strip()
        )


        code_series = (
            original
            .str.replace(
                ".0",
                "",
                regex=False
            )
            .str.zfill(2)
        )


        out["產業別"] = (
            code_series
            .map(
                產業對照
            )
        )


        中文 = (
            out[
                "產業別"
            ].isna()

            &

            original
            .str.contains(
                r"[\u4e00-\u9fff]",
                regex=True,
                na=False
            )
        )


        out.loc[
            中文,
            "產業別"
        ] = original[
            中文
        ]


    else:

        out["產業別"] = np.nan


    out["產業別"] = (
        out[
            "產業別"
        ]
        .astype("object")
        .fillna(
            "未分類"
        )
        .astype(str)
        .str.strip()
    )


    out["市場"] = market


    return out


全市場基本資料 = pd.concat(
    [
        整理公司母體(
            上市公司,
            "上市"
        ),

        整理公司母體(
            上櫃公司,
            "上櫃"
        )
    ],
    ignore_index=True
)


全市場基本資料 = 全市場基本資料[
    全市場基本資料[
        "股票代號"
    ].str.match(
        r"^[1-9]\d{3}$",
        na=False
    )
].copy()


全市場基本資料 = (
    全市場基本資料
    .drop_duplicates(
        "股票代號"
    )
    .reset_index(
        drop=True
    )
)


print(
    "全市場公司母體：",
    len(
        全市場基本資料
    )
)


# ============================================================
# 20. MOPS月營收
# ============================================================

上市營收 = 取得CSV(
    "https://mopsfin.twse.com.tw/"
    "opendata/t187ap05_L.csv"
)


上櫃營收 = 取得CSV(
    "https://mopsfin.twse.com.tw/"
    "opendata/t187ap05_O.csv"
)


def 整理月營收(df):

    if df.empty:
        return pd.DataFrame()


    code = 找欄位(
        df,
        ["公司代號"]
    )


    revenue = 找欄位(
        df,
        ["當月營收"]
    )


    last_month = 找欄位(
        df,
        ["上月營收"]
    )


    last_year = 找欄位(
        df,
        ["去年當月營收"]
    )


    yoy = 找欄位(
        df,
        ["去年同月增減"]
    )


    mom = 找欄位(
        df,
        ["上月增減"]
    )


    if code is None:
        return pd.DataFrame()


    out = pd.DataFrame()


    out["股票代號"] = (
        df[code]
        .astype(str)
        .str.strip()
    )


    if revenue:

        out["最新月營收"] = (
            df[revenue]
            .apply(數字)
        )


    if last_month:

        out["上月營收"] = (
            df[last_month]
            .apply(數字)
        )


    if last_year:

        out["去年同月營收"] = (
            df[last_year]
            .apply(數字)
        )


    if yoy:

        out["月營收年增率"] = (
            df[yoy]
            .apply(數字)
        )


    if mom:

        out["月營收月增率"] = (
            df[mom]
            .apply(數字)
        )


    if (
        "月營收年增率"
        not in out.columns
        and
        "最新月營收"
        in out.columns
        and
        "去年同月營收"
        in out.columns
    ):

        out["月營收年增率"] = (
            (
                安全除法(
                    out[
                        "最新月營收"
                    ],
                    out[
                        "去年同月營收"
                    ]
                )
                -
                1
            )
            *
            100
        )


    if (
        "月營收月增率"
        not in out.columns
        and
        "最新月營收"
        in out.columns
        and
        "上月營收"
        in out.columns
    ):

        out["月營收月增率"] = (
            (
                安全除法(
                    out[
                        "最新月營收"
                    ],
                    out[
                        "上月營收"
                    ]
                )
                -
                1
            )
            *
            100
        )


    return (
        out
        .drop_duplicates(
            "股票代號"
        )
    )


月營收資料 = pd.concat(
    [
        整理月營收(
            上市營收
        ),

        整理月營收(
            上櫃營收
        )
    ],
    ignore_index=True
)


月營收資料 = (
    月營收資料
    .drop_duplicates(
        "股票代號"
    )
)


# ============================================================
# 21. 財報格式
# ============================================================

財報格式 = {

    "ci":
        "一般業",

    "basi":
        "金融業",

    "bd":
        "證券期貨業",

    "fh":
        "金控業",

    "ins":
        "保險業",

    "mim":
        "異業"
}


# ============================================================
# 22. 損益表
# ============================================================

def 整理損益表(
    df,
    type_name
):

    if df.empty:
        return pd.DataFrame()


    code = 找欄位(
        df,
        ["公司代號"]
    )


    year = 找欄位(
        df,
        ["年度"]
    )


    quarter = 找欄位(
        df,
        ["季別"]
    )


    eps = 找欄位(
        df,
        [
            "基本每股盈餘（元）",
            "基本每股盈餘",
            "每股盈餘"
        ]
    )


    if type_name == "一般業":

        revenue = 找欄位(
            df,
            ["營業收入"]
        )


    elif type_name == "金融業":

        revenue = 找欄位(
            df,
            [
                "淨收益",
                "利息淨收益"
            ]
        )


    elif type_name == "證券期貨業":

        revenue = 找欄位(
            df,
            ["收益"]
        )


    elif type_name == "金控業":

        revenue = 找欄位(
            df,
            ["淨收益"]
        )


    elif type_name == "異業":

        revenue = 找欄位(
            df,
            ["收入"]
        )


    else:

        # 保險業不強行使用錯誤營收欄位
        revenue = None


    gross = 找欄位(
        df,
        [
            "營業毛利（毛損）淨額",
            "營業毛利（毛損）",
            "營業毛利"
        ]
    )


    operating = 找欄位(
        df,
        [
            "營業利益（損失）",
            "營業利益"
        ]
    )


    net_income = 找欄位(
        df,
        [
            "本期稅後淨利（淨損）",
            "本期淨利（淨損）",
            "本期淨利",
            "繼續營業單位本期稅後淨利（淨損）",
            "繼續營業單位本期淨利（淨損）"
        ]
    )


    if code is None:
        return pd.DataFrame()


    out = pd.DataFrame()


    out["股票代號"] = (
        df[code]
        .astype(str)
        .str.strip()
    )


    out["財報類型"] = (
        type_name
    )


    out["財報年度"] = (
        df[year]
        .apply(數字)
        if year
        else np.nan
    )


    out["財報季別"] = (
        df[quarter]
        .apply(數字)
        if quarter
        else np.nan
    )


    mapping = {

        "每股盈餘":
            eps,

        "財報營收":
            revenue,

        "營業毛利":
            gross,

        "營業利益":
            operating,

        "稅後淨利":
            net_income
    }


    for new_col, source_col in (
        mapping.items()
    ):

        if source_col:

            out[new_col] = (
                df[source_col]
                .apply(數字)
            )

        else:

            out[new_col] = np.nan


    # 一般業才套一般公司的毛利率 / 營業利益率
    if type_name == "一般業":

        out["毛利率"] = (
            安全除法(
                out[
                    "營業毛利"
                ],
                out[
                    "財報營收"
                ]
            )
            *
            100
        )


        out["營業利益率"] = (
            安全除法(
                out[
                    "營業利益"
                ],
                out[
                    "財報營收"
                ]
            )
            *
            100
        )


    else:

        out["毛利率"] = np.nan
        out["營業利益率"] = np.nan


    return out


損益列表 = []


print(
    "\n下載 MOPS 損益表..."
)


for market_code, market_name in [
    ("L", "上市"),
    ("O", "上櫃")
]:

    for type_code, type_name in (
        財報格式.items()
    ):

        url = (
            "https://mopsfin.twse.com.tw/"
            "opendata/"
            f"t187ap06_{market_code}_{type_code}.csv"
        )


        temp = 取得CSV(
            url
        )


        print(
            market_name,
            type_name,
            len(temp)
        )


        if not temp.empty:

            normalized = 整理損益表(
                temp,
                type_name
            )


            if not normalized.empty:

                損益列表.append(
                    normalized
                )


if not 損益列表:

    raise RuntimeError(
        "MOPS損益表全部下載失敗"
    )


損益資料 = pd.concat(
    損益列表,
    ignore_index=True,
    sort=False
)


損益資料["財報年度"] = pd.to_numeric(
    損益資料[
        "財報年度"
    ],
    errors="coerce"
)


損益資料["財報季別"] = pd.to_numeric(
    損益資料[
        "財報季別"
    ],
    errors="coerce"
)


損益資料 = (
    損益資料
    .sort_values(
        [
            "股票代號",
            "財報年度",
            "財報季別"
        ]
    )
    .drop_duplicates(
        "股票代號",
        keep="last"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 23. 資產負債表
# ============================================================

def 整理資產負債表(
    df,
    type_name
):

    if df.empty:
        return pd.DataFrame()


    code = 找欄位(
        df,
        ["公司代號"]
    )


    year = 找欄位(
        df,
        ["年度"]
    )


    quarter = 找欄位(
        df,
        ["季別"]
    )


    assets = 找欄位(
        df,
        [
            "資產總計",
            "資產總額"
        ]
    )


    liabilities = 找欄位(
        df,
        [
            "負債總計",
            "負債總額"
        ]
    )


    equity = 找欄位(
        df,
        [
            "權益總計",
            "權益總額"
        ]
    )


    current_assets = 找欄位(
        df,
        ["流動資產"]
    )


    current_liabilities = 找欄位(
        df,
        ["流動負債"]
    )


    if code is None:
        return pd.DataFrame()


    out = pd.DataFrame()


    out["股票代號"] = (
        df[code]
        .astype(str)
        .str.strip()
    )


    out["財報年度_資產"] = (
        df[year]
        .apply(數字)
        if year
        else np.nan
    )


    out["財報季別_資產"] = (
        df[quarter]
        .apply(數字)
        if quarter
        else np.nan
    )


    mapping = {

        "總資產":
            assets,

        "總負債":
            liabilities,

        "股東權益":
            equity,

        "流動資產":
            current_assets,

        "流動負債":
            current_liabilities
    }


    for new_col, source_col in (
        mapping.items()
    ):

        if source_col:

            out[new_col] = (
                df[source_col]
                .apply(數字)
            )

        else:

            out[new_col] = np.nan


    out["負債比率"] = (
        安全除法(
            out[
                "總負債"
            ],
            out[
                "總資產"
            ]
        )
        *
        100
    )


    if type_name not in [
        "金融業",
        "保險業",
        "金控業",
        "異業"
    ]:

        out["流動比率"] = (
            安全除法(
                out[
                    "流動資產"
                ],
                out[
                    "流動負債"
                ]
            )
            *
            100
        )


    else:

        out["流動比率"] = np.nan


    return out


資產列表 = []


print(
    "\n下載 MOPS 資產負債表..."
)


for market_code, market_name in [
    ("L", "上市"),
    ("O", "上櫃")
]:

    for type_code, type_name in (
        財報格式.items()
    ):

        url = (
            "https://mopsfin.twse.com.tw/"
            "opendata/"
            f"t187ap07_{market_code}_{type_code}.csv"
        )


        temp = 取得CSV(
            url
        )


        print(
            market_name,
            type_name,
            len(temp)
        )


        if not temp.empty:

            normalized = 整理資產負債表(
                temp,
                type_name
            )


            if not normalized.empty:

                資產列表.append(
                    normalized
                )


if not 資產列表:

    raise RuntimeError(
        "MOPS資產負債表全部下載失敗"
    )


資產資料 = pd.concat(
    資產列表,
    ignore_index=True,
    sort=False
)


資產資料["財報年度_資產"] = pd.to_numeric(
    資產資料[
        "財報年度_資產"
    ],
    errors="coerce"
)


資產資料["財報季別_資產"] = pd.to_numeric(
    資產資料[
        "財報季別_資產"
    ],
    errors="coerce"
)


資產資料 = (
    資產資料
    .sort_values(
        [
            "股票代號",
            "財報年度_資產",
            "財報季別_資產"
        ]
    )
    .drop_duplicates(
        "股票代號",
        keep="last"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 24. TWSE估值
# ============================================================

def 取得上市估值(date):

    url = (
        "https://www.twse.com.tw/"
        "rwd/zh/afterTrading/BWIBBU_d"
    )


    params = {

        "response":
            "json",

        "date":
            date.strftime("%Y%m%d"),

        "selectType":
            "ALL"
    }


    try:

        data = 取得JSON(
            url,
            params
        )


        df = pd.DataFrame(
            data.get(
                "data",
                []
            ),
            columns=
                data.get(
                    "fields",
                    []
                )
        )


        if df.empty:
            return pd.DataFrame()


        code = 找欄位(
            df,
            ["證券代號"]
        )


        pe = 找欄位(
            df,
            ["本益比"]
        )


        pb = 找欄位(
            df,
            ["股價淨值比"]
        )


        dy = 找欄位(
            df,
            ["殖利率"]
        )


        if code is None:
            return pd.DataFrame()


        out = pd.DataFrame()


        out["股票代號"] = (
            df[code]
            .astype(str)
            .str.strip()
        )


        out["本益比"] = (
            df[pe]
            .apply(數字)
            if pe
            else np.nan
        )


        out["股價淨值比"] = (
            df[pb]
            .apply(數字)
            if pb
            else np.nan
        )


        out["殖利率"] = (
            df[dy]
            .apply(數字)
            if dy
            else np.nan
        )


        return out


    except Exception as e:

        print(
            "上市估值取得失敗：",
            str(e)[:100]
        )

        return pd.DataFrame()


# ============================================================
# 25. TPEx估值
# ============================================================

def 取得上櫃估值(date):

    url = (
        "https://www.tpex.org.tw/"
        "web/stock/aftertrading/"
        "peratio_analysis/"
        "pera_result.php"
    )


    params = {

        "l":
            "zh-tw",

        "o":
            "json",

        "d":
            民國日期(date),

        "c":
            "",

        "s":
            "0,asc"
    }


    try:

        data = 取得JSON(
            url,
            params
        )


        result = []


        for row in data.get(
            "aaData",
            []
        ):

            if len(row) < 7:
                continue


            result.append({

                "股票代號":
                    str(row[0]).strip(),

                "本益比":
                    數字(row[2]),

                "殖利率":
                    數字(row[5]),

                "股價淨值比":
                    數字(row[6])
            })


        return pd.DataFrame(
            result
        )


    except Exception as e:

        print(
            "上櫃估值取得失敗：",
            str(e)[:100]
        )

        return pd.DataFrame()


上市估值 = 取得上市估值(
    最新交易日
)


上櫃估值 = 取得上櫃估值(
    最新交易日
)


估值資料 = pd.concat(
    [
        上市估值,
        上櫃估值
    ],
    ignore_index=True
)


估值資料 = (
    估值資料
    .drop_duplicates(
        "股票代號"
    )
)


# ============================================================
# 26. 建立全市場基本面
# ============================================================

全市場基本面 = (
    全市場基本資料
    .copy()
)


for df in [
    月營收資料,
    損益資料,
    資產資料,
    估值資料
]:

    if not df.empty:

        全市場基本面 = (
            全市場基本面
            .merge(
                df,
                on="股票代號",
                how="left"
            )
        )


# ============================================================
# 27. ROE / 總資產周轉率
#
# 目前延續已成功版本：
# 使用期末權益 / 期末資產
# 並將累計財報年化
#
# ============================================================

q = pd.to_numeric(
    全市場基本面[
        "財報季別"
    ],
    errors="coerce"
)


年化倍數 = (
    4 / q
).replace(
    [
        np.inf,
        -np.inf
    ],
    np.nan
)


全市場基本面[
    "ROE"
] = (
    安全除法(
        全市場基本面[
            "稅後淨利"
        ],
        全市場基本面[
            "股東權益"
        ]
    )
    *
    100
    *
    年化倍數
)


年化營收 = (
    全市場基本面[
        "財報營收"
    ]
    *
    年化倍數
)


全市場基本面[
    "總資產周轉率"
] = (
    安全除法(
        年化營收,
        全市場基本面[
            "總資產"
        ]
    )
)


# 不適用一般營運週轉率的產業
全市場基本面.loc[
    全市場基本面[
        "財報類型"
    ].isin(
        [
            "金融業",
            "保險業",
            "證券期貨業",
            "金控業",
            "異業"
        ]
    ),
    "總資產周轉率"
] = np.nan


# 這兩項暫不使用錯誤近似值
全市場基本面[
    "存貨周轉率"
] = np.nan


全市場基本面[
    "應收帳款周轉率"
] = np.nan


# ============================================================
# 28. 清理估值
# ============================================================

for col in [
    "本益比",
    "股價淨值比"
]:

    全市場基本面.loc[
        pd.to_numeric(
            全市場基本面[col],
            errors="coerce"
        )
        <=
        0,
        col
    ] = np.nan


# ============================================================
# 29. 同產業PR函數
# ============================================================

def 計算同產業PR(
    df,
    原始欄位,
    PR欄位,
    越高越好=True
):

    df = df.copy()


    df[PR欄位] = np.nan


    if 原始欄位 not in df.columns:
        return df


    df[原始欄位] = pd.to_numeric(
        df[原始欄位],
        errors="coerce"
    )


    df["產業別"] = (
        df["產業別"]
        .astype("object")
        .fillna(
            "未分類"
        )
        .astype(str)
        .str.strip()
    )


    if 越高越好:

        全市場PR = (
            df[
                原始欄位
            ]
            .rank(
                pct=True,
                ascending=True,
                method="average"
            )
            *
            100
        )


    else:

        全市場PR = (
            df[
                原始欄位
            ]
            .rank(
                pct=True,
                ascending=False,
                method="average"
            )
            *
            100
        )


    for industry, index in (
        df.groupby(
            "產業別",
            observed=False
        )
        .groups
        .items()
    ):

        series = df.loc[
            index,
            原始欄位
        ]


        sample_size = (
            series
            .notna()
            .sum()
        )


        if (
            industry != "未分類"
            and
            sample_size
            >=
            最低產業樣本數
        ):

            if 越高越好:

                pr = (
                    series
                    .rank(
                        pct=True,
                        ascending=True,
                        method="average"
                    )
                    *
                    100
                )


            else:

                pr = (
                    series
                    .rank(
                        pct=True,
                        ascending=False,
                        method="average"
                    )
                    *
                    100
                )


            df.loc[
                index,
                PR欄位
            ] = pr


        else:

            df.loc[
                index,
                PR欄位
            ] = (
                全市場PR.loc[
                    index
                ]
            )


    return df


# ============================================================
# 30. V3 基本面PR
# ============================================================

PR設定 = [

    (
        "每股盈餘",
        "每股盈餘PR",
        True
    ),

    (
        "ROE",
        "ROE PR",
        True
    ),

    (
        "毛利率",
        "毛利率PR",
        True
    ),

    (
        "營業利益率",
        "營業利益率PR",
        True
    ),

    (
        "月營收年增率",
        "月營收年增率PR",
        True
    ),

    (
        "月營收月增率",
        "月營收月增率PR",
        True
    ),

    (
        "負債比率",
        "負債比率PR",
        False
    ),

    (
        "流動比率",
        "流動比率PR",
        True
    ),

    (
        "總資產周轉率",
        "總資產周轉率PR",
        True
    ),

    (
        "本益比",
        "本益比PR",
        False
    ),

    (
        "股價淨值比",
        "股價淨值比PR",
        False
    ),

    (
        "殖利率",
        "殖利率PR",
        True
    )
]


for original, pr_col, higher_better in (
    PR設定
):

    全市場基本面 = 計算同產業PR(
        全市場基本面,
        original,
        pr_col,
        higher_better
    )


# PER + PBR 合成一項估值PR
全市場基本面[
    "估值PR"
] = (
    全市場基本面[
        [
            "本益比PR",
            "股價淨值比PR"
        ]
    ]
    .mean(
        axis=1,
        skipna=True
    )
)


全市場基本面[
    "同產業公司數"
] = (
    全市場基本面
    .groupby(
        "產業別"
    )[
        "股票代號"
    ]
    .transform(
        "count"
    )
)


# ============================================================
# 31. V3 基本面權重
#
# 目前沿用你已成功跑過的版本
# 滿分100
# ============================================================

基本面權重 = {

    "每股盈餘PR":
        12,

    "ROE PR":
        18,

    "毛利率PR":
        8,

    "營業利益率PR":
        12,

    "月營收年增率PR":
        14,

    "月營收月增率PR":
        6,

    "負債比率PR":
        7,

    "流動比率PR":
        5,

    "總資產周轉率PR":
        6,

    "估值PR":
        8,

    "殖利率PR":
        4
}


def 計算基本面分數(row):

    score = 0
    available_weight = 0


    for col, weight in (
        基本面權重.items()
    ):

        if (
            col in row.index
            and
            pd.notna(
                row[col]
            )
        ):

            score += (
                row[col]
                /
                100
                *
                weight
            )


            available_weight += weight


    if available_weight == 0:
        return np.nan


    return (
        score
        /
        available_weight
        *
        100
    )


全市場基本面[
    "基本面分數"
] = (
    全市場基本面
    .apply(
        計算基本面分數,
        axis=1
    )
)


# ============================================================
# 32. V3 基本面原因
# ============================================================

def 基本面原因(row):

    result = []


    mapping = {

        "每股盈餘PR":
            "EPS",

        "ROE PR":
            "ROE",

        "毛利率PR":
            "毛利率",

        "營業利益率PR":
            "營業利益率",

        "月營收年增率PR":
            "營收年增",

        "月營收月增率PR":
            "營收月增",

        "負債比率PR":
            "負債比",

        "流動比率PR":
            "流動比率",

        "總資產周轉率PR":
            "資產周轉",

        "估值PR":
            "估值",

        "殖利率PR":
            "殖利率"
    }


    for col, name in (
        mapping.items()
    ):

        if (
            col in row.index
            and
            pd.notna(
                row[col]
            )
            and
            row[col] >= 80
        ):

            result.append(
                f"{name}同產業前20%"
            )


    return "、".join(
        result
    )


全市場基本面[
    "基本面原因"
] = (
    全市場基本面
    .apply(
        基本面原因,
        axis=1
    )
)


# ============================================================
# 33. V3資料完整度
# ============================================================

print(
    "\n======================================"
)

print(
    "V3 資料完整度"
)

print(
    "======================================"
)


for col in [

    "每股盈餘",
    "ROE",
    "毛利率",
    "營業利益率",
    "月營收年增率",
    "月營收月增率",
    "負債比率",
    "流動比率",
    "總資產周轉率",
    "本益比",
    "股價淨值比",
    "殖利率",
    "基本面分數"

]:

    print(
        f"{col}：",
        全市場基本面[
            col
        ].notna().sum(),
        "/",
        len(
            全市場基本面
        )
    )


# ============================================================
# 34. 合併V1 + V4
# ============================================================

技術資料 = (
    最新技術資料
    .rename(
        columns={

            "stock_id":
                "股票代號",

            "close":
                "收盤價"
        }
    )
)


結果 = (
    活躍股票[
        [
            "股票代號",
            "股票名稱",
            "市場",
            "成交股數"
        ]
    ]
    .copy()
)


結果 = 結果.merge(
    技術資料,
    on="股票代號",
    how="left"
)


# ============================================================
# 35. 合併V2
# ============================================================

if not 法人摘要.empty:

    結果 = 結果.merge(
        法人摘要,
        on="股票代號",
        how="left"
    )


法人欄位 = [

    "外資近5日買賣超股數",
    "投信近5日買賣超股數",
    "自營商近5日買賣超股數",
    "三大法人近5日合計股數",
    "外資連續買超天數",
    "投信連續買超天數",
    "法人資料天數"
]


for col in 法人欄位:

    if col not in 結果.columns:

        結果[col] = 0


    結果[col] = (
        pd.to_numeric(
            結果[col],
            errors="coerce"
        )
        .fillna(0)
    )


# ============================================================
# 36. V2籌碼評分
# ============================================================

結果[
    "籌碼分數"
] = 0


結果[
    "籌碼面原因"
] = ""


def 籌碼加分(
    condition,
    score,
    reason
):

    結果.loc[
        condition,
        "籌碼分數"
    ] += score


    結果.loc[
        condition,
        "籌碼面原因"
    ] += reason + "、"


籌碼加分(
    結果[
        "外資近5日買賣超股數"
    ]
    >
    0,
    10,
    "外資近5日買超"
)


籌碼加分(
    結果[
        "投信近5日買賣超股數"
    ]
    >
    0,
    15,
    "投信近5日買超"
)


籌碼加分(
    結果[
        "自營商近5日買賣超股數"
    ]
    >
    0,
    5,
    "自營商近5日買超"
)


籌碼加分(
    結果[
        "外資連續買超天數"
    ]
    >=
    3,
    10,
    "外資連買至少3日"
)


籌碼加分(
    結果[
        "投信連續買超天數"
    ]
    >=
    3,
    15,
    "投信連買至少3日"
)


籌碼加分(
    結果[
        "三大法人近5日合計股數"
    ]
    >
    0,
    10,
    "三大法人合計買超"
)


結果[
    "籌碼面原因"
] = (
    結果[
        "籌碼面原因"
    ]
    .str.rstrip(
        "、"
    )
)


# ============================================================
# 37. 合併V3
# ============================================================

V3欄位 = [

    "股票代號",

    "產業別",
    "同產業公司數",
    "財報類型",

    "每股盈餘",
    "每股盈餘PR",

    "ROE",
    "ROE PR",

    "毛利率",
    "毛利率PR",

    "營業利益率",
    "營業利益率PR",

    "月營收年增率",
    "月營收年增率PR",

    "月營收月增率",
    "月營收月增率PR",

    "負債比率",
    "負債比率PR",

    "流動比率",
    "流動比率PR",

    "總資產周轉率",
    "總資產周轉率PR",

    "存貨周轉率",
    "應收帳款周轉率",

    "本益比",
    "本益比PR",

    "股價淨值比",
    "股價淨值比PR",

    "估值PR",

    "殖利率",
    "殖利率PR",

    "最新月營收",

    "基本面分數",
    "基本面原因"
]


for col in V3欄位:

    if col not in 全市場基本面.columns:

        全市場基本面[
            col
        ] = np.nan


結果 = 結果.merge(
    全市場基本面[
        V3欄位
    ],
    on="股票代號",
    how="left"
)


# ============================================================
# 38. 股數轉張數
# ============================================================

結果[
    "當日成交量（張）"
] = (
    結果[
        "成交股數"
    ]
    /
    1000
)


for source, target in [

    (
        "外資近5日買賣超股數",
        "外資近5日買賣超（張）"
    ),

    (
        "投信近5日買賣超股數",
        "投信近5日買賣超（張）"
    ),

    (
        "自營商近5日買賣超股數",
        "自營商近5日買賣超（張）"
    ),

    (
        "三大法人近5日合計股數",
        "三大法人近5日合計（張）"
    )

]:

    結果[target] = (
        結果[source]
        /
        1000
    )


# ============================================================
# 39. V5 籌碼標準化
#
# 65分 → 100分
# ============================================================

print(
    "\n======================================"
)

print(
    "V5 最終決策層"
)

print(
    "======================================"
)


結果[
    "籌碼標準分"
] = (
    pd.to_numeric(
        結果[
            "籌碼分數"
        ],
        errors="coerce"
    )
    /
    65
    *
    100
)


結果[
    "籌碼標準分"
] = (
    結果[
        "籌碼標準分"
    ]
    .clip(
        lower=0,
        upper=100
    )
)


# ============================================================
# 40. V5 最終分數
#
# V1 30%
# V2 20%
# V3 30%
# V4 20%
# ============================================================

for col in [
    "技術分數",
    "籌碼標準分",
    "基本面分數",
    "風險動能分數"
]:

    結果[col] = pd.to_numeric(
        結果[col],
        errors="coerce"
    )


結果[
    "最終分數"
] = (

    結果[
        "技術分數"
    ]
    *
    0.30

    +

    結果[
        "籌碼標準分"
    ]
    *
    0.20

    +

    結果[
        "基本面分數"
    ]
    *
    0.30

    +

    結果[
        "風險動能分數"
    ]
    *
    0.20
)


結果[
    "最終分數"
] = (
    結果[
        "最終分數"
    ]
    .round(2)
)


# ============================================================
# 41. V5 綜合PR
# ============================================================

結果[
    "綜合PR"
] = (
    結果[
        "最終分數"
    ]
    .rank(
        pct=True,
        ascending=True,
        method="average"
    )
    *
    100
)


結果[
    "綜合PR"
] = (
    結果[
        "綜合PR"
    ]
    .round(1)
)


# ============================================================
# 42. V5 候選等級
# ============================================================

def V5候選等級(row):

    技術 = row[
        "技術分數"
    ]

    基本 = row[
        "基本面分數"
    ]

    V4 = row[
        "風險動能分數"
    ]

    PR = row[
        "綜合PR"
    ]


    if (
        pd.notna(PR)
        and
        PR >= 90

        and

        pd.notna(技術)
        and
        技術 >= 70

        and

        pd.notna(基本)
        and
        基本 >= 60

        and

        pd.notna(V4)
        and
        V4 >= 55
    ):

        return "★★★★★ 強勢候選"


    elif (
        pd.notna(PR)
        and
        PR >= 75

        and

        pd.notna(技術)
        and
        技術 >= 60

        and

        pd.notna(V4)
        and
        V4 >= 45
    ):

        return "★★★★ 觀察候選"


    elif (
        pd.notna(PR)
        and
        PR >= 50
    ):

        return "★★★ 一般觀察"


    else:

        return "★★ 弱勢"


結果[
    "候選等級"
] = (
    結果
    .apply(
        V5候選等級,
        axis=1
    )
)


# ============================================================
# 43. V5目前狀態
# ============================================================

def 判斷目前狀態(row):

    技術 = row.get(
        "技術分數",
        np.nan
    )

    基本 = row.get(
        "基本面分數",
        np.nan
    )

    V4 = row.get(
        "風險動能分數",
        np.nan
    )

    報酬5 = row.get(
        "5日報酬率",
        np.nan
    )

    報酬20 = row.get(
        "20日報酬率",
        np.nan
    )

    乖離20 = row.get(
        "MA20乖離率",
        np.nan
    )

    回撤 = row.get(
        "20日最大回撤",
        np.nan
    )

    ATR = row.get(
        "ATR百分比",
        np.nan
    )

    波動 = row.get(
        "20日年化波動率",
        np.nan
    )


    # 短線過熱
    if (
        (
            pd.notna(乖離20)
            and
            乖離20 >= 15
        )

        or

        (
            pd.notna(報酬5)
            and
            報酬5 >= 18
        )

        or

        (
            pd.notna(報酬20)
            and
            報酬20 >= 35
        )
    ):

        return "短線過熱"


    # 風險偏高
    if (
        (
            pd.notna(回撤)
            and
            回撤 <= -20
        )

        or

        (
            pd.notna(ATR)
            and
            ATR >= 7
        )

        or

        (
            pd.notna(波動)
            and
            波動 >= 65
        )
    ):

        return "風險偏高"


    # 強勢但乖離偏高
    if (
        pd.notna(技術)
        and
        技術 >= 70

        and

        pd.notna(乖離20)
        and
        乖離20 >= 8
    ):

        return "等待回檔"


    # 健康強勢
    if (
        pd.notna(技術)
        and
        技術 >= 70

        and

        pd.notna(V4)
        and
        V4 >= 60
    ):

        return "趨勢健康"


    # 基本面強、技術未發動
    if (
        pd.notna(基本)
        and
        基本 >= 60

        and

        (
            pd.isna(技術)
            or
            技術 < 70
        )
    ):

        return "整理觀察"


    return "一般觀察"


結果[
    "目前狀態"
] = (
    結果
    .apply(
        判斷目前狀態,
        axis=1
    )
)


# ============================================================
# 44. V5綜合理由
# ============================================================

def 產生V5理由(row):

    reasons = []


    技術 = row.get(
        "技術分數",
        np.nan
    )


    if pd.notna(技術):

        if 技術 >= 80:

            reasons.append(
                "技術趨勢強"
            )

        elif 技術 >= 70:

            reasons.append(
                "技術面偏多"
            )


    籌碼 = row.get(
        "籌碼標準分",
        np.nan
    )


    if pd.notna(籌碼):

        if 籌碼 >= 75:

            reasons.append(
                "法人籌碼強"
            )

        elif 籌碼 >= 50:

            reasons.append(
                "法人偏多"
            )


    基本 = row.get(
        "基本面分數",
        np.nan
    )


    if pd.notna(基本):

        if 基本 >= 80:

            reasons.append(
                "同產業基本面突出"
            )

        elif 基本 >= 65:

            reasons.append(
                "基本面優於產業平均"
            )


    V4 = row.get(
        "風險動能分數",
        np.nan
    )


    if pd.notna(V4):

        if V4 >= 75:

            reasons.append(
                "風險動能結構佳"
            )

        elif V4 >= 60:

            reasons.append(
                "風險動能健康"
            )


    PR = row.get(
        "綜合PR",
        np.nan
    )


    if pd.notna(PR):

        if PR >= 90:

            reasons.append(
                "綜合表現前10%"
            )

        elif PR >= 75:

            reasons.append(
                "綜合表現前25%"
            )


    return "、".join(
        reasons
    )


結果[
    "V5綜合理由"
] = (
    結果
    .apply(
        產生V5理由,
        axis=1
    )
)


# ============================================================
# 45. V5風險提示
# ============================================================

def 產生風險提示(row):

    warning_list = []


    報酬5 = row.get(
        "5日報酬率",
        np.nan
    )


    報酬20 = row.get(
        "20日報酬率",
        np.nan
    )


    乖離20 = row.get(
        "MA20乖離率",
        np.nan
    )


    乖離60 = row.get(
        "MA60乖離率",
        np.nan
    )


    回撤 = row.get(
        "20日最大回撤",
        np.nan
    )


    ATR = row.get(
        "ATR百分比",
        np.nan
    )


    波動 = row.get(
        "20日年化波動率",
        np.nan
    )


    volume_cv = row.get(
        "20日成交量變異係數",
        np.nan
    )


    RSI = row.get(
        "14日RSI",
        np.nan
    )


    if (
        pd.notna(報酬5)
        and
        報酬5 >= 18
    ):

        warning_list.append(
            "5日漲幅過大"
        )


    if (
        pd.notna(報酬20)
        and
        報酬20 >= 35
    ):

        warning_list.append(
            "20日漲幅過大"
        )


    if (
        pd.notna(乖離20)
        and
        乖離20 >= 15
    ):

        warning_list.append(
            "MA20乖離過大"
        )


    if (
        pd.notna(乖離60)
        and
        乖離60 >= 30
    ):

        warning_list.append(
            "MA60乖離過大"
        )


    if (
        pd.notna(ATR)
        and
        ATR >= 7
    ):

        warning_list.append(
            "ATR偏高"
        )


    if (
        pd.notna(波動)
        and
        波動 >= 65
    ):

        warning_list.append(
            "波動率偏高"
        )


    if (
        pd.notna(回撤)
        and
        回撤 <= -20
    ):

        warning_list.append(
            "近期回撤偏大"
        )


    if (
        pd.notna(volume_cv)
        and
        volume_cv > 1
    ):

        warning_list.append(
            "成交量波動較大"
        )


    if (
        pd.notna(RSI)
        and
        RSI >= 75
    ):

        warning_list.append(
            "RSI偏熱"
        )


    if not warning_list:

        return "無明顯風險訊號"


    return "、".join(
        warning_list
    )


結果[
    "V5風險提示"
] = (
    結果
    .apply(
        產生風險提示,
        axis=1
    )
)


# ============================================================
# 46. V5 最終排名
# ============================================================

結果 = (
    結果
    .sort_values(
        [
            "最終分數",
            "綜合PR",
            "技術分數",
            "基本面分數",
            "風險動能分數",
            "籌碼標準分"
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
            False
        ],
        na_position="last"
    )
    .reset_index(
        drop=True
    )
)


if "排名" in 結果.columns:

    結果 = 結果.drop(
        columns=[
            "排名"
        ]
    )


結果.insert(
    0,
    "排名",
    range(
        1,
        len(結果) + 1
    )
)


# ============================================================
# 47. 最終診斷
# ============================================================

print(
    "\n======================================"
)

print(
    "V1 + V2 + V3 + V4 + V5 最終檢查"
)

print(
    "======================================"
)


print(
    "成交量篩選股票：",
    len(結果)
)


print(
    "有技術分數：",
    結果[
        "技術分數"
    ].notna().sum()
)


print(
    "有籌碼分數：",
    結果[
        "籌碼分數"
    ].notna().sum()
)


print(
    "有籌碼標準分：",
    結果[
        "籌碼標準分"
    ].notna().sum()
)


print(
    "有基本面分數：",
    結果[
        "基本面分數"
    ].notna().sum()
)


print(
    "有V4風險動能分數：",
    結果[
        "風險動能分數"
    ].notna().sum()
)


print(
    "有V5最終分數：",
    結果[
        "最終分數"
    ].notna().sum()
)


print(
    "\n最高V4分數：",
    結果[
        "風險動能分數"
    ].max()
)


print(
    "最高最終分數：",
    round(
        結果[
            "最終分數"
        ].max(),
        2
    )
)


print(
    "最低最終分數：",
    round(
        結果[
            "最終分數"
        ].min(),
        2
    )
)


print(
    "平均最終分數：",
    round(
        結果[
            "最終分數"
        ].mean(),
        2
    )
)


print(
    "\n候選等級分布："
)


print(
    結果[
        "候選等級"
    ]
    .value_counts()
)


print(
    "\n目前狀態分布："
)


print(
    結果[
        "目前狀態"
    ]
    .value_counts()
)


print(
    "\nFinMind 批次呼叫次數：",
    FinMind批次呼叫次數
)


print(
    "FinMind 單股補抓次數：",
    FinMind補抓次數
)


print(
    "是否遇到FinMind額度上限：",
    "是"
    if FinMind額度中止
    else "否"
)


print(
    "======================================"
)


# ============================================================
# 48. Excel 主表欄位
# ============================================================

輸出欄位 = [

    # V5最重要
    "排名",

    "市場",
    "產業別",

    "股票代號",
    "股票名稱",

    "最終分數",
    "綜合PR",

    "候選等級",
    "目前狀態",

    "技術分數",
    "籌碼標準分",
    "基本面分數",
    "風險動能分數",

    "V5綜合理由",
    "V5風險提示",

    # 公司資訊
    "財報類型",
    "同產業公司數",

    # V1
    "收盤價",

    "5日均線",
    "20日均線",
    "60日均線",

    "14日RSI",

    "MACD",
    "MACD訊號線",
    "MACD柱狀體",

    "量比",

    "當日成交量（張）",

    # V2
    "籌碼分數",

    "外資近5日買賣超（張）",
    "投信近5日買賣超（張）",
    "自營商近5日買賣超（張）",
    "三大法人近5日合計（張）",

    "外資連續買超天數",
    "投信連續買超天數",

    "法人資料天數",

    # V3
    "每股盈餘",
    "每股盈餘PR",

    "ROE",
    "ROE PR",

    "毛利率",
    "毛利率PR",

    "營業利益率",
    "營業利益率PR",

    "月營收年增率",
    "月營收年增率PR",

    "月營收月增率",
    "月營收月增率PR",

    "負債比率",
    "負債比率PR",

    "流動比率",
    "流動比率PR",

    "總資產周轉率",
    "總資產周轉率PR",

    "存貨周轉率",
    "應收帳款周轉率",

    "本益比",
    "本益比PR",

    "股價淨值比",
    "股價淨值比PR",

    "估值PR",

    "殖利率",
    "殖利率PR",

    "最新月營收",

    # V4
    "5日報酬率",
    "20日報酬率",
    "60日報酬率",

    "20日年化波動率",

    "ATR14",
    "ATR百分比",

    "20日最大回撤",

    "MA20乖離率",
    "MA60乖離率",

    "20日成交量變異係數",

    # 原因
    "技術面原因",
    "籌碼面原因",
    "基本面原因",
    "V4原因"
]


for col in 輸出欄位:

    if col not in 結果.columns:

        結果[col] = np.nan


輸出資料 = (
    結果[
        輸出欄位
    ]
    .copy()
)


# ============================================================
# 49. 強勢候選
# ============================================================

強勢候選資料 = (
    輸出資料[
        輸出資料[
            "候選等級"
        ]
        ==
        "★★★★★ 強勢候選"
    ]
    .copy()
)


# ============================================================
# 50. 強勢 + 觀察候選
# ============================================================

候選股票資料 = (
    輸出資料[
        輸出資料[
            "候選等級"
        ]
        .isin(
            [
                "★★★★★ 強勢候選",
                "★★★★ 觀察候選"
            ]
        )
    ]
    .copy()
)


# ============================================================
# 51. 前20名
# ============================================================

前20名資料 = (
    輸出資料
    .head(
        前幾名
    )
    .copy()
)


# ============================================================
# 52. V5決策摘要
# ============================================================

V5摘要欄位 = [

    "排名",

    "股票代號",
    "股票名稱",
    "產業別",

    "最終分數",
    "綜合PR",

    "候選等級",
    "目前狀態",

    "技術分數",
    "籌碼標準分",
    "基本面分數",
    "風險動能分數",

    "收盤價",

    "5日報酬率",
    "20日報酬率",
    "60日報酬率",

    "MA20乖離率",
    "20日最大回撤",

    "ATR百分比",
    "20日年化波動率",

    "14日RSI",

    "V5綜合理由",
    "V5風險提示"
]


V5摘要資料 = (
    結果[
        V5摘要欄位
    ]
    .copy()
)


# ============================================================
# 53. V4專屬工作表
# ============================================================

V4欄位 = [

    "排名",

    "股票代號",
    "股票名稱",
    "產業別",

    "風險動能分數",

    "目前狀態",

    "收盤價",

    "5日報酬率",
    "20日報酬率",
    "60日報酬率",

    "20日年化波動率",

    "ATR14",
    "ATR百分比",

    "20日最大回撤",

    "MA20乖離率",
    "MA60乖離率",

    "20日成交量變異係數",

    "V4原因",

    "V5風險提示"
]


V4資料 = (
    結果[
        V4欄位
    ]
    .copy()
)


# ============================================================
# 54. 全市場PR工作表
# ============================================================

全市場PR欄位 = [

    "股票代號",
    "公司名稱",
    "市場",
    "產業別",
    "財報類型",
    "同產業公司數",

    "每股盈餘",
    "每股盈餘PR",

    "ROE",
    "ROE PR",

    "毛利率",
    "毛利率PR",

    "營業利益率",
    "營業利益率PR",

    "月營收年增率",
    "月營收年增率PR",

    "月營收月增率",
    "月營收月增率PR",

    "負債比率",
    "負債比率PR",

    "流動比率",
    "流動比率PR",

    "總資產周轉率",
    "總資產周轉率PR",

    "本益比",
    "本益比PR",

    "股價淨值比",
    "股價淨值比PR",

    "估值PR",

    "殖利率",
    "殖利率PR",

    "基本面分數"
]


for col in 全市場PR欄位:

    if col not in 全市場基本面.columns:

        全市場基本面[
            col
        ] = np.nan


全市場PR資料 = (
    全市場基本面[
        全市場PR欄位
    ]
    .copy()
)


# ============================================================
# 55. 產業平均與中位數
# ============================================================

統計指標 = [

    "每股盈餘",
    "ROE",
    "毛利率",
    "營業利益率",
    "月營收年增率",
    "月營收月增率",
    "負債比率",
    "流動比率",
    "總資產周轉率",
    "本益比",
    "股價淨值比",
    "殖利率"
]


正式產業資料 = (
    全市場基本面[
        全市場基本面[
            "產業別"
        ]
        !=
        "未分類"
    ]
    .copy()
)


產業平均 = (
    正式產業資料
    .groupby(
        "產業別"
    )[
        統計指標
    ]
    .mean()
    .add_suffix(
        "－平均"
    )
)


產業中位數 = (
    正式產業資料
    .groupby(
        "產業別"
    )[
        統計指標
    ]
    .median()
    .add_suffix(
        "－中位數"
    )
)


產業家數 = (
    正式產業資料
    .groupby(
        "產業別"
    )[
        "股票代號"
    ]
    .count()
    .rename(
        "產業公司數"
    )
)


產業統計 = (
    pd.concat(
        [
            產業家數,
            產業平均,
            產業中位數
        ],
        axis=1
    )
    .reset_index()
)


# ============================================================
# 56. V5權重說明
# ============================================================

V5權重說明 = pd.DataFrame({

    "構面": [
        "V1 技術面",
        "V2 籌碼面",
        "V3 基本面",
        "V4 風險動能"
    ],

    "原始滿分": [
        100,
        65,
        100,
        100
    ],

    "V5處理": [
        "直接使用",
        "除以65乘100",
        "直接使用",
        "直接使用"
    ],

    "V5權重": [
        "30%",
        "20%",
        "30%",
        "20%"
    ]
})


# ============================================================
# 57. 系統說明
# ============================================================

系統說明 = pd.DataFrame({

    "項目": [

        "資料基準日",

        "第一層股票池",

        "V1技術面",

        "V2籌碼面",

        "V3基本面",

        "V4風險動能",

        "V5最終分數",

        "綜合PR",

        "強勢候選",

        "觀察候選"
    ],

    "說明": [

        str(
            最新交易日
        ),

        "4碼上市櫃普通股、排除00開頭ETF、當日成交量至少1000張",

        "MA、RSI、MACD、量比、突破",

        "外資、投信、自營商近5日與連買",

        "MOPS財報與月營收＋官方估值＋同產業PR",

        "報酬、波動率、ATR、回撤、乖離、成交量穩定",

        "技術30%＋籌碼20%＋基本面30%＋風險動能20%",

        "在當日成交量股票池中的最終分數百分位",

        "PR>=90、技術>=70、基本面>=60、V4>=55",

        "PR>=75、技術>=60、V4>=45"
    ]
})


# ============================================================
# 58. Excel輸出
# ============================================================

檔名 = (
    "台股V1V2V3V4V5最終選股_"
    +
    最新交易日.strftime(
        "%Y%m%d"
    )
    +
    ".xlsx"
)


with pd.ExcelWriter(
    檔名,
    engine="openpyxl"
) as writer:


    輸出資料.to_excel(
        writer,
        sheet_name=
            "全部排名",
        index=False
    )


    強勢候選資料.to_excel(
        writer,
        sheet_name=
            "強勢候選",
        index=False
    )


    候選股票資料.to_excel(
        writer,
        sheet_name=
            "候選股票",
        index=False
    )


    前20名資料.to_excel(
        writer,
        sheet_name=
            "前20名",
        index=False
    )


    V5摘要資料.to_excel(
        writer,
        sheet_name=
            "V5決策摘要",
        index=False
    )


    V4資料.to_excel(
        writer,
        sheet_name=
            "V4風險動能",
        index=False
    )


    全市場PR資料.to_excel(
        writer,
        sheet_name=
            "全市場產業PR",
        index=False
    )


    產業統計.to_excel(
        writer,
        sheet_name=
            "產業平均與中位數",
        index=False
    )


    V5權重說明.to_excel(
        writer,
        sheet_name=
            "V5權重說明",
        index=False
    )


    系統說明.to_excel(
        writer,
        sheet_name=
            "系統說明",
        index=False
    )


# ============================================================
# 59. Excel美化
# ============================================================

wb = load_workbook(
    檔名
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"

    ws.auto_filter.ref = (
        ws.dimensions
    )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )


    ws.row_dimensions[
        1
    ].height = 24


    for column_cells in (
        ws.columns
    ):

        max_length = 0


        letter = get_column_letter(
            column_cells[0].column
        )


        for cell in column_cells:

            value = (
                ""
                if cell.value is None
                else str(
                    cell.value
                )
            )


            max_length = max(
                max_length,
                len(value)
            )


        ws.column_dimensions[
            letter
        ].width = min(
            max_length + 2,
            38
        )


wb.save(
    檔名
)


# ============================================================
# 60. 顯示結果
# ============================================================

print(
    "\n✅ V1～V5 Excel 已建立：",
    檔名
)


print(
    "\n======================================"
)

print(
    "V5 最終排名前20名"
)

print(
    "======================================"
)


print(
    結果[
        [
            "排名",
            "股票代號",
            "股票名稱",
            "產業別",
            "最終分數",
            "綜合PR",
            "候選等級",
            "目前狀態",
            "技術分數",
            "籌碼標準分",
            "基本面分數",
            "風險動能分數"
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)


# ============================================================
# 61. Colab自動下載
# ============================================================

try:

    from google.colab import files

    files.download(
        檔名
    )

except:

    pass

請輸入 FinMind Token：··········


2026-09-27 03:26:37.256 | INFO     | FinMind.data.finmind_api:login_by_token:89 - Login success



最新交易日： 2026-09-24
成交量 >= 1000張： 344 檔

V1 + V4 共用 FinMind 歷史資料


2026-09-27 03:26:37.310 | INFO     | FinMind.data.finmind_api:login_by_token:89 - Login success
2026-09-27 03:26:37.312 | INFO     | FinMind.data.finmind_api:_get_data_with_async:207 - download Dataset.TaiwanStockPrice, data_id: ['2409', '2303', '3481', '6770', '8150', '1303', '2344', '3016', '1709', '3576', '1727', '3189', '2342', '1314', '3042', '6116', '2408', '1101', '2327', '2481', '2317', '3231', '2426', '2449', '2610', '4956', '3094', '2887', '2883', '4958']


股價批次 1 ~ 30 / 344


...

股價批次 331 ~ 344 / 344


100%|██████████| 14/14 [00:00<00:00, 41.35it/s]



批次已抓股票： 344
需要補抓股票： 0

歷史資料成功股票： 344 / 344

V1 技術面

V4 風險動能

V2 籌碼面
法人交易日： 2026-09-24
法人交易日： 2026-09-23
法人交易日： 2026-09-22
法人交易日： 2026-09-21
法人交易日： 2026-09-18

V3 MOPS 基本面
全市場公司母體： 1982

下載 MOPS 損益表...
上市 一般業 1049
上市 金融業 10
上市 證券期貨業 3
上市 金控業 13
上市 保險業 5
上市 異業 4
上櫃 一般業 885
上櫃 金融業 1
上櫃 證券期貨業 7
上櫃 金控業 1
上櫃 保險業 1
上櫃 異業 1

下載 MOPS 資產負債表...
上市 一般業 1049
上市 金融業 10
上市 證券期貨業 3
上市 金控業 13
上市 保險業 5
上市 異業 4
上櫃 一般業 885
上櫃 金融業 1
上櫃 證券期貨業 7
上櫃 金控業 1
上櫃 保險業 1
上櫃 異業 1

V3 資料完整度
每股盈餘： 1976 / 1982
ROE： 1976 / 1982
毛利率： 1933 / 1982
營業利益率： 1933 / 1982
月營收年增率： 1971 / 1982
月營收月增率： 1968 / 1982
負債比率： 1976 / 1982
流動比率： 1944 / 1982
總資產周轉率： 1934 / 1982
本益比： 871 / 1982
股價淨值比： 1081 / 1982
殖利率： 1081 / 1982
基本面分數： 1978 / 1982

V5 最終決策層

V1 + V2 + V3 + V4 + V5 最終檢查
成交量篩選股票： 344
有技術分數： 344
有籌碼分數： 344
有籌碼標準分： 344
有基本面分數： 344
有V4風險動能分數： 344
有V5最終分數： 344

最高V4分數： 100
最高最終分數： 81.83
最低最終分數： 15.2
平均最終分數： 47.11

候選等級分布：
候選等級
★★ 弱勢         171
★★★ 一般觀察      120
★★★★ 觀察候選      36
★★★★★ 強勢候選     17
Name: count, dtype: int64

目前狀態分

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# ============================================================
# 額外輸出 App 用歷史技術資料
# ============================================================

App歷史資料 = 歷史資料.copy()

App歷史資料 = App歷史資料.rename(
    columns={
        "stock_id": "股票代號",
        "date": "日期",
        "open": "開盤價",
        "max": "最高價",
        "min": "最低價",
        "close": "收盤價",
        "Trading_Volume": "成交股數"
    }
)

保留欄位 = [
    "股票代號",
    "日期",

    "開盤價",
    "最高價",
    "最低價",
    "收盤價",
    "成交股數",

    "5日均線",
    "20日均線",
    "60日均線",

    "14日RSI",

    "MACD",
    "MACD訊號線",
    "MACD柱狀體",

    "量比",

    "5日報酬率",
    "20日報酬率",
    "60日報酬率",

    "20日年化波動率",

    "ATR14",
    "ATR百分比",

    "20日最大回撤",

    "MA20乖離率",
    "MA60乖離率",

    "20日成交量變異係數"
]

保留欄位 = [
    col
    for col in 保留欄位
    if col in App歷史資料.columns
]

App歷史資料 = App歷史資料[
    保留欄位
].copy()

App歷史資料["股票代號"] = (
    App歷史資料["股票代號"]
    .astype(str)
    .str.zfill(4)
)

App歷史資料 = App歷史資料.sort_values(
    [
        "股票代號",
        "日期"
    ]
)

歷史檔名 = "history_latest.csv"

App歷史資料.to_csv(
    歷史檔名,
    index=False,
    encoding="utf-8-sig"
)

print(
    "✅ App歷史資料已建立：",
    歷史檔名
)

try:
    from google.colab import files
    files.download(歷史檔名)
except:
    pass

✅ App歷史資料已建立： history_latest.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>